In [ ]:
import dataclasses

import jax
import numpy as np

from openpi.models import model as _model
from openpi.policies import pika_policy
from openpi.policies import policy_config as _policy_config
from openpi.shared import download
from openpi.training import config as _config
from openpi.training import data_loader as _data_loader

from zmq_bridge import zmq_image_bridge
from zmq_bridge import zmq_joint_pos_bridge
from zmq_bridge import zmq_joint_cmd_bridge

import time
import matplotlib.pyplot as plt

In [ ]:
camera_receiver = zmq_image_bridge.ZMQImageReceiver("tcp://localhost:5555", resize=(224, 224), encoding = "rgb8")
fisheye_receiver = zmq_image_bridge.ZMQImageReceiver("tcp://localhost:5556", resize=(224, 224))
joint_pos_receiver = zmq_joint_pos_bridge.ZMQJointPosReceiver("tcp://localhost:5557")
joint_cmd_sender = zmq_joint_cmd_bridge.ZMQJointCmdSender("tcp://localhost:6001")

config_name = "pi0_pika"
episode_num = "66"
train_setp = "9999"

config = _config.get_config(config_name)
checkpoint_dir = "/mnt/hdd/openpi/checkpoints/" + config_name + "_" + episode_num + "_episode/" + train_setp
print(checkpoint_dir)
policy = _policy_config.create_trained_policy(config, checkpoint_dir)




In [ ]:
joint_cmd_sender.send_array(np.array([0.1,0.2,-0.2,0.0,0.8,0.0,0.01]))

In [ ]:
while True:
    camera_img = camera_receiver.receive_once()
    fisheye_img = fisheye_receiver.receive_once()
    joint_pos = joint_pos_receiver.receive_once()
    print(joint_pos)
    task = "put the fluorescent green ball into the spotted mug"

    pika_input = {
        "observation/state": joint_pos,
        "observation/image": fisheye_img,
        "observation/wrist_image": camera_img,
        "task": task,
    }
    result = policy.infer(pika_input)

    for row in result["actions"][:20]:
        joint_cmd_sender.send_array(row)
        time.sleep(1/30)



# plt.figure(figsize=(6, 3))

# plt.subplot(1, 2, 1)
# plt.title("Camera Image")
# plt.imshow(camera_img)
# plt.axis("off")

# plt.subplot(1, 2, 2)
# plt.title("Fisheye Image")
# plt.imshow(fisheye_img)
# plt.axis("off")

# plt.tight_layout()
# plt.show()

In [ ]:
del policy

joint_cmd_sender.close()
camera_receiver.close()
fisheye_receiver.close()
joint_pos_receiver.close()

# Working with a live model


The following example shows how to create a live model from a checkpoint and compute training loss. First, we are going to demonstrate how to do it with fake data.


In [ ]:
config = _config.get_config("pi0_aloha_sim")

checkpoint_dir = download.maybe_download("gs://openpi-assets/checkpoints/pi0_aloha_sim")
key = jax.random.key(0)

# Create a model from the checkpoint.
model = config.model.load(_model.restore_params(checkpoint_dir / "params"))

# We can create fake observations and actions to test the model.
obs, act = config.model.fake_obs(), config.model.fake_act()

# Sample actions from the model.
loss = model.compute_loss(key, obs, act)
print("Loss shape:", loss.shape)

Now, we are going to create a data loader and use a real batch of training data to compute the loss.

In [ ]:
# Reduce the batch size to reduce memory usage.
config = dataclasses.replace(config, batch_size=2)

# Load a single batch of data. This is the same data that will be used during training.
# NOTE: In order to make this example self-contained, we are skipping the normalization step
# since it requires the normalization statistics to be generated using `compute_norm_stats`.
loader = _data_loader.create_data_loader(config, num_batches=1, skip_norm_stats=True)
obs, act = next(iter(loader))

# Sample actions from the model.
loss = model.compute_loss(key, obs, act)

# Delete the model to free up memory.
del model

print("Loss shape:", loss.shape)